|<h2>Course:</h2>|<h1><a href="https://udemy.com/course/dullms_x/?couponCode=202508" target="_blank">A deep understanding of AI language model mechanisms</a></h1>|
|-|:-:|
|<h2>Part 2:</h2>|<h1>Large language models<h1>|
|<h2>Section:</h2>|<h1>Build a GPT<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge: Do we really need Q?<b></h1>|

<br>

<h5><b>Teacher:</b> Mike X Cohen, <a href="https://sincxpress.com" target="_blank">sincxpress.com</a></h5>
<h5><b>Course URL:</b> <a href="https://udemy.com/course/dullms_x/?couponCode=202508" target="_blank">udemy.com/course/dullms_x/?couponCode=202508</a></h5>
<i>Using the code without the course may lead to confusion or errors.</i>

In [ ]:
import torch
from transformers import AutoModelForCausalLM,GPT2Tokenizer

# Exercise 1: Import GPT2 and push a copy to the GPU

In [ ]:
# load SMALL pretrained GPT-2 model and tokenizer
gpt2_orig = AutoModelForCausalLM.from_pretrained('gpt2') # this one won't be modified
gpt2      = AutoModelForCausalLM.from_pretrained('gpt2')
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')

In [ ]:
# use GPU
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(device)

In [ ]:
# push the model to the GPU
gpt2.to(device)

In [ ]:
# generate some input
x = torch.tensor(tokenizer.encode('Hello, how are you today?')).unsqueeze(0)

In [ ]:
### on the CPU
torch.manual_seed(42) # seed the rng
out = gpt2_orig.generate(x,temperature=1,do_sample=True,max_length=100,
                    pad_token_id=tokenizer.eos_token_id)
print(tokenizer.decode(out[0].tolist()))

In [ ]:
### on the GPU
torch.manual_seed(42) # seed the rng
out = gpt2.generate(x.to(device),temperature=1,do_sample=True,max_length=100,
                    pad_token_id=tokenizer.eos_token_id)
print(tokenizer.decode(out[0].tolist()))

# Note: some differences between CPU and GPU are expected due to differential rounding/precision errors
#       but the rng seed is different on the GPU and CPU

In [ ]:
# FYI, use this code to copy the original parameters onto a modified model
gpt2.load_state_dict( gpt2_orig.state_dict() )

# Exercise 2: Replace one Q matrix

In [ ]:
# size of Q matrix
n_emb = gpt2.transformer.wte.weight.shape[1]
sizeOfQ = gpt2.transformer.h[0].attn.c_attn.weight[:,:n_emb].shape
sizeOfQ

In [ ]:
# get the matrix
q = gpt2.transformer.h[0].attn.c_attn.weight[:,:n_emb]

# create noise with matching statistics
mean = q.mean()
std  = q.std()
noise = torch.randn_like(q)*std + mean
      # Note: 
      #   torch.randn_like(q) generate normally distributed
      #     numbers, that has same size as q, with mean 0 and standard deviation 1.
      #     so you need to multiply it by std and add mean.


# replace the values in q
gpt2.transformer.h[0].attn.c_attn.weight.data[:,:n_emb] = noise
# q.data.copy_(noise); # this also works...

In [ ]:
# confirm that the weights really have changed...
print(gpt2.transformer.h[0].attn.c_attn.weight[:10,1])
print(gpt2_orig.transformer.h[0].attn.c_attn.weight[:10,1])

In [ ]:
### re-run on the GPU
torch.manual_seed(42) # seed the rng
out = gpt2.generate(x.to(device),temperature=1,do_sample=True,max_length=100,
                    pad_token_id=tokenizer.eos_token_id)
print(tokenizer.decode(out[0].tolist()))

# Exercise 3: Successively lobotomize the model :(

In [ ]:
# reset the model to its original parameters
gpt2.load_state_dict( gpt2_orig.state_dict() )

In [ ]:
# model output pre-changes
print('** Using all original Q values:')
x = torch.tensor(tokenizer.encode('I went to the market to buy')).unsqueeze(0)
out = gpt2.generate(x.to(device),temperature=1,do_sample=True,max_length=50,
                    pad_token_id=tokenizer.eos_token_id)
print(tokenizer.decode(out[0].tolist()))


# loop over all transformer heads, replace Q, and give the same input
for hidx in range(len(gpt2.transformer.h)):

  print(f'\n\n** Now replacing Q{hidx} with noise:')

  # get the matrix
  q = gpt2.transformer.h[hidx].attn.c_attn.weight[:,:n_emb]

  # create noise with matching statistics
  mean = q.mean()
  std  = q.std()
  noise = torch.randn_like(q)*std + mean

  # replace the values in q
  gpt2.transformer.h[hidx].attn.c_attn.weight.data[:,:n_emb] = noise

  # generate the output
  out = gpt2.generate(x.to(device),temperature=1,do_sample=True,max_length=50,
                      pad_token_id=tokenizer.eos_token_id)
  print(tokenizer.decode(out[0].tolist()))